In [ ]:
# Problema: Elegir una representación de datos adecuada comparando CSV, JSON y Parquet sobre el mismo subconjunto real de vuelos.
from pathlib import Path
import pandas as pd
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'data').is_dir() and (p / 'submission').is_dir())
DATA, OUTPUT = ROOT / 'data', ROOT / 'submission/format_comparison.csv'


In [ ]:
# La comparación es válida porque los tres formatos parten de las mismas filas.
frame = pd.read_csv(DATA / 'flights.csv.gz')
frame.shape, frame.dtypes.head()


In [ ]:
csv_path, json_path, parquet_path = DATA / 'flights.csv', DATA / 'flights.json', DATA / 'flights.parquet'
frame.to_csv(csv_path, index=False)
frame.to_json(json_path, orient='records')
frame.to_parquet(parquet_path, index=False, engine='pyarrow')


In [ ]:
# Antes de comparar tamaños, comprobamos que CSV y Parquet conservan el mismo número de filas.
assert len(frame) == len(pd.read_csv(csv_path)) == len(pd.read_parquet(parquet_path, engine='pyarrow'))
[(path.name, path.stat().st_size) for path in [csv_path, json_path, parquet_path]]


In [ ]:
comparison = pd.DataFrame([['CSV', csv_path.stat().st_size, False, True, False, 'Intercambio e inspección simple'], ['JSON', json_path.stat().st_size, False, True, False, 'Datos semiestructurados e intercambio'], ['Parquet', parquet_path.stat().st_size, True, False, True, 'Lectura analítica columnar']], columns=['format', 'file_size_bytes', 'schema_preserved', 'human_readable', 'column_selection', 'primary_use_case'])
comparison.to_csv(OUTPUT, index=False)
comparison.sort_values('file_size_bytes')
